


Contributions:

Part 1. Main Contributor text: Lovro Antic s235263
    - Subcontributor: Uffe Grøn Roepstorff s245109


Part 2. Main Contributor code and text: Oskar Karlsson s244771
    -Subcontributor: Lovro Antic s235263

Part 3, Main Contributor code and text.: Uffe Grøn Roepstorff s245109
    -Subcontributor: Oskar Karlsson s244771


In [5]:
import requests
import pandas as pd
import time
from tqdm import tqdm
import unicodedata
import re
from bs4 import BeautifulSoup
from joblib import Parallel, delayed

## Part 1

#### **Pros and Cons of the Custom-made Vs Ready-made Data**


Centola's experiment used custom-made data from a controlled platform, ensuring a clean, "unpolluted" dataset. While the artificial environment allowed for precise variable manipulation, it incurred high costs and risked swaying data due to its lack of organic development. However, because participants were unaware of the specific monitoring, typical response biases were minimized.

Conversely, Nicolaides's study utilized ready-made data from a long-standing app. This provided a massive, inexpensive sample size representative of real-world behavior. The primary disadvantage was "dirty" data; since the information wasn't fine-tuned for research, it lacked vital context like weather or emotional states.



#### **Different Influence**

Centola's Study: Because the environment was custom-built, we can trust the causal links. We know exactly why a behavior spread because the "noise" was removed. However, we must interpret the results with a grain of salt, would people act the same way on a messy, distracted platform like Facebook or X? It proves what can happen in a vacuum.

Nicolaides's Study: This offers ecological validity. The results show what actually happens in our daily lives. However, because the data is "ready-made," the interpretation is more about correlation. Without knowing if it was a sunny day or if a user felt motivated, we can't be 100% certain the app's social features were the only cause of the behavior.

# Part 2

### Retrieve data

In [6]:
EMAIL = 'okf2003@gmail.com' # Other account mail's244771@dtu.dk'
BASE_URL_AUTHORS = 'https://api.openalex.org/authors'
BASE_URL_WORKS = 'https://api.openalex.org/works'
KEY = '1yqALT6NyxS775KrS65fLl' # Other account ket: 'YU1NMyYLC0SJYghyehfP0i'

In [7]:
df = pd.read_csv('authors.csv') # list of authors from week 1

names = set() # identify unique authors in the author dataset
for entry in df['author'].dropna():
    for name in entry.split(','):
        name = name.strip()
        if name:
            names.add(name)

names = list(names)

print(f'Total unique authors: {len(names)}')

Total unique authors: 1565


### Helper functions, created iteratively after encountering problems

In [8]:
def normalize_name(name):
    # Normalize the name to remove accents and special characters
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    return name

def fallback_name(name):
    # Remove middle initials like "A." or "B.C."
    return re.sub(r'\b[A-Z]\.\s*', '', name).strip()

### Search each name in OpenAlex and retrieve information

In [ ]:
# Initialize lists to store author information
author_ids = []
display_names = []
works_api_urls = []
h_indexes = []
works_counts = []
country_codes = []


for name in tqdm(names):
    counter = 0
    params = {
        'search': normalize_name(name),
        'mailto': EMAIL,
        'api_key': KEY
    }
    response = requests.get('https://api.openalex.org/authors', params=params).json()
    result = response.get('results', [])

    if not result: # Fallback method: Try to remove the middle initials and search again
        retry_name = fallback_name(normalize_name(name))
        if retry_name != normalize_name(name): # Only retry if the name was actually modified
            params['search'] = retry_name
            response = requests.get('https://api.openalex.org/authors', params=params).json()
            result = response.get('results', [])


    if result:
        author = result[0] # Take the first result as the most relevant match
        author_ids.append(result[0]['id'])
        display_names.append(result[0]['display_name'])
        works_api_urls.append(result[0]['works_api_url'])
        h_indexes.append(author.get('summary_stats', {}).get('h_index')) # Inside summary_stats dict
        works_counts.append(result[0]['works_count'])
        # Get country code from last known institution
        institution = author.get('last_known_institutions', [])
        country_code = institution[0].get('country_code') if institution else None
        country_codes.append(country_code)
    else:
        author_ids.append(None)
        display_names.append(None)
        works_api_urls.append(None)
        h_indexes.append(None)
        works_counts.append(None)
        country_codes.append(None)

100%|██████████| 1565/1565 [08:02<00:00,  3.24it/s]


### Store in dataframe

In [10]:
# Store the collected data in a DataFrame
authors_df = pd.DataFrame({
    'author_id': author_ids,
    'display_name': display_names,
    'works_api_url': works_api_urls,
    'h_index': h_indexes,
    'works_count': works_counts,
    'country_code': country_codes
})

# strip url from author_id to get the clean actual id
authors_df['author_id'] = authors_df['author_id'].str.replace('https://openalex.org/', '', regex=False)

In [11]:
# Remove rows with missing author_id and save to csv
authors_df = authors_df.dropna(subset=['author_id'])
authors_df.to_csv('part1_authors_info.csv', index=False)
authors_df

,author_id,display_name,works_api_url,h_index,works_count,country_code
0,A5089975070,Katarzyna Sznajd-Weron,https://api.openalex.org/works?filter=author.i...,24.0,116.0,PL
1,A5015866712,Ekaterina Landgren,https://api.openalex.org/works?filter=author.i...,1.0,10.0,US
2,A5063313700,Deji Akinwande,https://api.openalex.org/works?filter=author.i...,74.0,486.0,US
3,A5078609284,AmirEmad Ghassami,https://api.openalex.org/works?filter=author.i...,11.0,66.0,NaN
4,A5022698469,Heetae Kim,https://api.openalex.org/works?filter=author.i...,22.0,126.0,KR
...,...,...,...,...,...,...
989,A5006672000,Heiko Rauhut,https://api.openalex.org/works?filter=author.i...,16.0,83.0,NaN
991,A5073672615,Grant Schoenebeck,https://api.openalex.org/works?filter=author.i...,19.0,119.0,US
992,A5040974704,Christian Bretter,https://api.openalex.org/works?filter=author.i...,10.0,33.0,AU
993,A5040873440,Anastasia Golovin,https://api.openalex.org/works?filter=author.i...,6.0,18.0,DE


### Reflection questions
Which challenges did you encounter? How did you address them?
We encountered several challenges while identifying the authors in OpenAlex, and here is a few examples:
1. API usage: Without proper queries you quickly run out of API credits. We adressed it by running tests on smaller batches of the dataset, still statistically representative of the dataset. We also created multiple accounts to get more credits
2. Not all the information we were to retrieve could be gathered directly from the api response. We resolved this by e.g. retrieving the authors last known institutions and therefrom the postal code. We had issues finding the h-index, but digging through our output dictionaires, we found it in the sub dictionary called 'summary stats'. 

Choose one problem you faced while collecting the data and describe your solution. Why did you choose this approach, and what impact might it have on your data?

Not all authors in the csv file could be found in OpenAlex by just making an API-call with their name. This was the main problem in this exercise and we solved it by created a set of helper functions. The first called normalize_name strips special characters from the name, e.g. Étienne --> Etienne. The second function called fallback_name removes middle initials from a name - names are not necesarily spelled with middle initial in OpenAlex. By implementing these two function, we reduced the percentage of missing authors in OpenAlex from 55% to 39%.

# Part 3 

In [12]:
API_KEY = 'U2HIfMm7lzzOACIZH2GvPS' # Uffe 'I7BU1yoA12r5RaYCvba8eC' # When we run out of keys

### Work Count filter

In [13]:
#There is no works count filter in the filtes, so we filter the works count before
#we find their works

mask = (authors_df['works_count'] > 5) & (authors_df['works_count'] < 5000)
authors_df = authors_df[mask]

### Works finder

In [ ]:
WORKING_URL = 'https://api.openalex.org/works'
EMAIL = "s245109@dtu.dk"  

#Defining topics.
social_ids = "C144024400|C15744967|C162324750|C17744445"
quant_ids = "C33923547|C121332964|C41008148"
BATCH_SIZE = 25

ids = authors_df['author_id'].tolist() #
#Creating a list of all our Batches
batches = [ids[i : i + BATCH_SIZE] for i in range(0, len(ids), BATCH_SIZE)]

#Creating workers.
def fetch_works_for_batch(batch_ids):
    batch_idstr = '|'.join(batch_ids)
    current_cursor = '*'
    batch_works = [] 
    
    filter_string = (
        f'author.id:{batch_idstr},'
        'authors_count:<10,'
        f'concepts.id:{social_ids},'
        f'concepts.id:{quant_ids},'
        'cited_by_count:>10'
    )

    while True:
        params = {
            'filter': filter_string,
            'select': 'id,publication_year,cited_by_count,authorships,title,abstract_inverted_index,concepts',
            'per_page': 200,
            'cursor': current_cursor,
            'mailto': EMAIL,
            'api_key': API_KEY 
        }

        try:
            response = requests.get(WORKING_URL, params=params)
            
            if response.status_code == 200:
                data = response.json()
                results = data.get('results', [])
                meta = data.get('meta', {})
                
                if not results:
                    break
                    
                for work in results:
                    author_ids = [
                        auth['author']['id'].split('/')[-1] if auth.get('author') and auth['author'].get('id')
                        else auth['author']['id']
                        for auth in work.get('authorships', [])
                    ]
                    batch_works.append({
                        'id': work.get('id'),
                        'publication_year': work.get('publication_year'),
                        'cited_by_count': work.get('cited_by_count'),
                        'title': work.get('title'),
                        'author_ids': author_ids, 
                        'abstract_inverted_index': work.get('abstract_inverted_index')
                    })
                
                current_cursor = meta.get('next_cursor')
                if not current_cursor:
                    break
                    
            elif response.status_code == 429:
                # If we hit the API request limit.
                time.sleep(2)
                continue 
            else:
                #Errors log.
                print(f"ERROR: {response.status_code}")
                break
                
        except Exception as e:
            #Backup for the work.
            time.sleep(2)
            continue

    #Returning workds
    return batch_works

# 3. Kør Joblib Parallel processen
print(f"Starter parallel download for {len(batches)} batches...")

#Retrieving n=4 jobs. 
results_nested = Parallel(n_jobs=4)(
    delayed(fetch_works_for_batch)(batch) for batch in tqdm(batches, desc="Collecting works")
)

# Flattening the list
all_works = [work for sublist in results_nested for work in sublist]

print(f"\nRetrieved {len(all_works)} works total.")

Starter parallel download for 33 batches...



Retrieved 10808 works total.


### Creation and export of DataFrames

In [18]:
#Creating the dataframes
D1 = pd.DataFrame(all_works)
D1.to_csv('Ex3_get_all_works.csv')
print(D1.shape, f'we have found {D1.shape[0]} works') 
D2 = D1[['id', 'publication_year', 'cited_by_count', 'author_ids']]
D3 = D1[['id', 'title', 'abstract_inverted_index']]
D2.to_csv('Ex3_D2.csv')
D3.to_csv('Ex3_D3.csv')

(10808, 6) we have found 10808 works


#### Unique researchers count

In [19]:
#Retrieving all the unique ID'
#Collecting all authors in a list
all_auth_ids = [author_id for sublist in D2['author_ids'] for author_id in sublist]
# Removing dupes via set.
unique_researchers = len(set(all_auth_ids))
print('Unique researchers found', unique_researchers)


Unique researchers found 15555


#### Dataset summaries:

##### **Dataset summary.** 
*How many works are listed in your Dataset D2 (IC2S2 papers) dataframe? How many unique researchers have co-authored these works?*




We have found **10371 works** with our filters,  as well a retrieveing **16201 unique authors**.


It's worth noting that this dataset hase some NaN values, see the section with NaN.
Many of these Nan's are attributed to the Abstract Inverted Index (AII) (2698 papers), but as the documentation notes is that quite common, as many older papers doesn't have an AII. 
Furthermore do we have one untitled paper and 12 papers with puplication years missing.

We have decided not to remove, or change any NaNs' yet, as we still could use the data to explore connectivity. (Author_Id's does NOT have any missing values, or empty lists)
Thus depending on the usecase these could be relevant. 


##### **Efficiency in code.** 

*Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?*


We optimized our code to maximize efficiency and minimize API requests using the following strategies:

1. Pre-filtering: Excluded authors outside the 5–5,000 works range before initiating any API calls.

2. Cursor Pagination: Utilized cursors to retrieve works efficiently per request.

3. Server-side Filtering: Applied complex API filters to download only relevant data.

4. Batching: Grouped 25 authors per request using the | (OR) operator, reducing total requests and conserving tokens.

5. Robustness: Implemented a pause condition to handle rate limits and prevent API overloads.

6. Parallel Processing: Executed requests concurrently across 4 cores. This step alone reduced the total execution time by a third.

Comparing the optimized work-retrieval with the unoptimized author retrieval, we see the major improvements. Retrieving 942 authors in 8 minutes vs 10808 works in 20 seconds.
Assuming the retrieval time is similar, the works process is 275 times faster.


#### **Filtering Criteria and Dataset Relevance** 
*Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?*

The filters we used help find relevant CSS works and researchers. Filtering works per author removes large institutions and irrelevant authors. Requiring 10+ citations helps us look only at influential work, although this limit could be arbitrary. Excluding projects with >10 authors mitigates megaprojects that are beyond the scope of just CSS research and could dominate. Concept IDs are also crucial, as there is no accurate CSS label, so works should cover both subjects.

One flaw of our method is that it prioritizes older, established researchers while excluding new researchers and PhDs, as they lack citations and output. There could also be many CSS papers labeled under only one field, e.g., only Economics. Lastly, more niche subfields could be skipped, as smaller fields have fewer citations.